# STAR AI ULTRA — NVIDIA A100 80GB / Qwen3.5 (Colab)
**الهدف:** تشغيل Qwen3.5 محلياً على GPU السحابي الخاص بجلسة Colab، دون مفتاح API خارجي، مع محادثة واختبار GPU وأوامر تشخيص Linux مقيدة.

- اختر من Colab: **Runtime → Change runtime type → GPU → A100 → High-RAM** (إذا كانت متاحة).
- شغّل الخلايا بالترتيب. تنزيل الأوزان كبير ويستهلك الشبكة والمساحة ووحدات Colab.
- يختار الدفتر نموذج **qwen3.5:35b** تلقائياً عندما تسمح الذاكرة والمساحة، وإلا ينتقل إلى 27B أو 9B أو 4B.
- لا يوجد منفذ منشور للإنترنت: Ollama يستمع إلى `127.0.0.1:11434` فقط.
- **Colab ليس خادماً دائماً.** قد تتوقف الجلسة أو يتغير نوع GPU. حافظ على ملفاتك في Drive أو GitHub، ولا تحفظ أي أسرار داخل الدفتر.
- الأدوات التشخيصية للقراءة فقط، ولا تنفذ أوامر اقترحها الذكاء الاصطناعي دون مراجعة بشرية.

**ملاحظة:** إنشاء الدفتر ونشره لا يثبت تنفيذ خلاياه؛ نجاح الاختبارات الحية يظهر فقط في مخرجات Colab.

## 1) تحقق فعلي من GPU والذاكرة والمساحة
يُرفض التشغيل دون NVIDIA GPU، وتُقرأ الموارد من الجلسة الحالية بدلاً من افتراض سعة ثابتة.

In [ ]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

def run_cmd(args, timeout=30):
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout, check=False)

gpu_check = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"])
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError("No NVIDIA GPU detected. Select Runtime → Change runtime type → GPU.")

gpu_name, total_mib, free_mib = [
    v.strip() for v in gpu_check.stdout.splitlines()[0].rsplit(",", 2)
]
gpu_free_gib = int(free_mib) / 1024
disk_free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print("GPU:", gpu_name)
print("GPU total GiB:", round(int(total_mib) / 1024, 1))
print("GPU free GiB:", round(gpu_free_gib, 1))
print("Disk free GiB:", round(disk_free_gib, 1))
print("Python:", sys.version.split()[0])
print("Runtime location: Colab cloud; your laptop is not executing this model.")

## 2) اختيار النموذج وفق الموارد الفعلية
الأحجام المطلوبة أدناه تقديرات محافظة للتنزيل وتشغيل سياق 8192، وليست ضماناً للموارد. اترك `MODEL_OVERRIDE=None` للاختيار الآلي. **35B** هو نموذج Qwen3.5 MoE بحجم تنزيل تقريبي 22GB حسب مكتبة Ollama.

In [ ]:
MODEL_OVERRIDE = None  # Example: "qwen3.5:27b" — only if you know available resources
MODEL_CHOICES = [
    ("qwen3.5:35b", 32, 32),  # GPU-free GiB, disk-free GiB (conservative)
    ("qwen3.5:27b", 26, 26),
    ("qwen3.5:9b", 12, 12),
    ("qwen3.5:4b", 7, 8),
]
eligible = [name for name, need_gpu, need_disk in MODEL_CHOICES
            if gpu_free_gib >= need_gpu and disk_free_gib >= need_disk]
if not eligible:
    raise RuntimeError("Not enough free GPU memory or disk to download any configured model.")
if MODEL_OVERRIDE is not None and MODEL_OVERRIDE not in eligible:
    raise ValueError(f"Override not supported by measured resources. Eligible: {eligible}")
MODEL = MODEL_OVERRIDE or eligible[0]
CONTEXT = 8192
print("Selected:", MODEL)
print("Eligible:", ", ".join(eligible))
print("No model has been downloaded yet.")

## 3) تثبيت Ollama من المصدر الرسمي
يحمّل السكربت الرسمي من `ollama.com`. افحص سياسة الثقة في مصادر التثبيت قبل تشغيله في أي بيئة مؤسسية.

In [ ]:
if shutil.which("ollama") is None:
    install_script = "/tmp/star-ai-ollama-install.sh"
    subprocess.run(["curl", "-fsSL", "https://ollama.com/install.sh",
                    "-o", install_script], check=True)
    subprocess.run(["bash", install_script], check=True)
if not shutil.which("ollama"):
    raise RuntimeError("Ollama installation failed")
print(run_cmd(["ollama", "--version"]).stdout.strip())

## 4) تشغيل الخادم داخل جلسة Colab فقط
لا تستخدم نفقاً عاماً أو رابطاً بدون مصادقة. تم ضبط مضيف Ollama على localhost حصراً.

In [ ]:
import requests

OLLAMA_HOST = "127.0.0.1:11434"
OLLAMA_URL = "http://" + OLLAMA_HOST
ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT),
    "OLLAMA_KEEP_ALIVE": "5m",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
})

def ollama_ready():
    try:
        return requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_ready():
    log_file = open("/tmp/star-ai-ultra-ollama.log", "a")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"], env=ollama_env,
        stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True
    )
    for _ in range(60):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            break
        time.sleep(1)
if not ollama_ready():
    tail = Path("/tmp/star-ai-ultra-ollama.log").read_text(errors="replace")[-2500:]
    raise RuntimeError("Ollama failed to start. Log tail:\n" + tail)
print("Local Ollama ready:", OLLAMA_URL)

## 5) تنزيل الأوزان (مرة لكل جلسة مؤقتة)
قد يستهلك نموذج 35B قرابة **22GB** من التنزيل. لا تُشغّل هذه الخلية إن كنت لا تريد استهلاك رصيد/حزمة إنترنت Colab. إذا لم تكفِ مساحة القرص، شغّل خلية الاختيار السابقة بعد ضبط `MODEL_OVERRIDE` إلى نموذج أصغر.

In [ ]:
print("Downloading model:", MODEL)
print("Available storage GiB before download:", round(shutil.disk_usage("/content").free / 2**30, 1))
subprocess.run(["ollama", "pull", MODEL], check=True, env=ollama_env)
print("Downloaded successfully:", MODEL)

## 6) اختبار محادثة حقيقي + إثبات هل التنفيذ على GPU
رسالة الرد وحدها لا تثبت استخدام GPU؛ يتم عرض `ollama ps` و`nvidia-smi` بعد تنفيذ الاستدلال.

In [ ]:
SYSTEM = (
    "You are STAR AI ULTRA, an Arabic-first coding assistant for software engineering "
    "and authorized defensive security tasks. Be exact; never claim to have run "
    "commands or accessed external systems unless a provided tool result proves it. "
    "For any impactful system change, require user approval."
)

def ask_model(messages, max_tokens=512):
    r = requests.post(
        OLLAMA_URL + "/api/chat",
        json={
            "model": MODEL, "messages": messages, "stream": False,
            "think": False,
            "options": {"num_ctx": CONTEXT, "num_predict": max_tokens,
                        "temperature": 0.35}, "keep_alive": "5m"
        },
        timeout=600
    )
    r.raise_for_status()
    payload = r.json()
    answer = (payload.get("message") or {}).get("content", "").strip()
    if not answer:
        raise RuntimeError("Model returned no text; inspect /tmp/star-ai-ultra-ollama.log")
    return answer, payload

reply, stats = ask_model([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "اكتب سطرين بالعربية عن تحليل سجلات Linux الدفاعي."}
], max_tokens=180)
print("MODEL RESPONSE:\n", reply)
print("\nMODEL:", stats.get("model"))
print("Generated tokens:", stats.get("eval_count"))
print("\nGPU/CPU processor report (check PROCESSOR column):")
print(subprocess.run(["ollama", "ps"], env=ollama_env,
                     capture_output=True, text=True).stdout)
print("\nCurrent GPU usage:")
print(run_cmd(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
               "--format=csv,noheader"]).stdout.strip())

## 7) محادثة تفاعلية مع STAR AI
اكتب `chat("سؤالك")` لتتحدث مع النموذج. حفظ الذاكرة هنا مؤقت داخل الجلسة وليس في GitHub أو Drive.

In [ ]:
history = [{"role": "system", "content": SYSTEM}]

def chat(question, max_tokens=1000):
    global history
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must not be empty")
    answer, _ = ask_model(
        history + [{"role": "user", "content": question.strip()}],
        max_tokens=max_tokens
    )
    history.extend([
        {"role": "user", "content": question.strip()},
        {"role": "assistant", "content": answer}
    ])
    history = [history[0]] + history[-10:]
    print("STAR AI ULTRA:\n" + answer)
    return answer

chat("اقترح هيكل مشروع Python لمراقبة السجلات والتنبيه الدفاعي مع اختبارات وحدات.", max_tokens=700)

## 8) تشخيص Linux الحقيقي — أوامر محددة وآمنة
هذه الأوامر تُنفّذ فعلياً على **خادم Colab**، وليس لابتوبك أو Railway. لا يُشغّل أي نص يأتي من النموذج كأمر shell.

In [ ]:
SAFE_DIAGNOSTICS = {
    "gpu": ["nvidia-smi"],
    "ram": ["free", "-h"],
    "disk": ["df", "-h", "/content"],
    "python": ["python", "--version"],
    "ollama": ["ollama", "ps"],
}

def diagnostic(name):
    if name not in SAFE_DIAGNOSTICS:
        raise ValueError("Allowed diagnostics: " + ", ".join(SAFE_DIAGNOSTICS))
    result = subprocess.run(SAFE_DIAGNOSTICS[name], shell=False,
                            capture_output=True, text=True, timeout=30,
                            env=ollama_env)
    print(result.stdout or result.stderr)
    return result.returncode

diagnostic("gpu")

## 9) حفظ محادثة غير حساسة اختيارياً في Google Drive
بشكل افتراضي **لا يتم تركيب Drive ولا حفظ محادثات**. إذا أردت الحفظ، غيّر `SAVE_TO_DRIVE=True`؛ سيطلب Colab تفويض حسابك. راجع المحتوى قبل الحفظ، ولا تحفظ بيانات عملاء أو مفاتيح API.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_dir = Path("/content/drive/MyDrive/STAR_AI_ULTRA")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "chat_history.json"
    output_file.write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", output_file)
else:
    print("Drive persistence disabled. No chat files written.")

## 10) إيقاف التشغيل والحماية
- عند الانتهاء: Colab → **Runtime → Disconnect and delete runtime** لإنهاء استهلاك GPU.
- لا تستخدم جلسة Colab لاستضافة API إنتاجي دائم. للنشر المستمر استخدم خادم GPU مخصصاً، HTTPS، مصادقة، صلاحيات محددة، وسجلات تدقيق.
- `OLLAMA_HOST=127.0.0.1` يعني أن Ollama داخل الجلسة فقط، ولا يمكن استدعاؤه مباشرة من Railway أو Cloudflare.
- للتحقق من تحميل الأوزان على GPU راجع عمود `PROCESSOR` في `ollama ps`؛ إذا كان يستخدم CPU فالاختبار لا يحقق هدف التسريع.
- إن ظهرت أخطاء تحميل أو نفاد الذاكرة: أوقف النموذج ب`ollama stop MODEL` من خلية Python، اختر نموذجاً أصغر، وأعد تشغيل خلايا التنزيل والاختبار. مخرجات كل خطوة هي دليل النجاح الفعلي.
- لا تضع مفاتيح API أو روابط حساسة في Notebook عام على GitHub.

### مصادر
- [Qwen3.5 على Ollama](https://ollama.com/library/qwen3.5)
- [توثيق Ollama API](https://docs.ollama.com/api)
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)